# 03 — Baseline Benchmark: Seasonal-Naive & WAPE Evaluation

**Deliverable D2 Companion Notebook**

This notebook implements the mandatory **Seasonal-Naive baseline ($t - 52$ weeks)** with a **3-tier fallback hierarchy** and establishes the official performance scorecard on rolling-origin cross-validation folds.

### Non-Negotiable Governance Adherence:
- **Rule 4 (Mandatory Seasonal-Naive Benchmark):** Any machine learning model must earn its deployment by beating this benchmark on identical backtest folds.
- **Rule 5 (Strictly Temporal Validation):** Rolling-origin CV preserving historical chronology; no random shuffling.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from foresight.config import load_settings
from foresight.forecasting.baseline import seasonal_naive_forecast, seasonal_naive_with_fallback
from foresight.evaluation.metrics import wape, bias, mae, rmse

settings = load_settings()
processed = settings.processed_dir
weekly = pd.read_parquet(processed / 'sales_weekly.parquet')
sku_master = pd.read_csv(settings.dataset_file('raw_sku'))
print(f"Loaded weekly panel: {weekly.shape[0]:,} rows across {weekly['sku_id'].nunique()} SKUs")

Loaded weekly panel: 1,050,000 rows across 5000 SKUs


## 1. The 3-Tier Fallback Hierarchy

To robustly handle zero-sales periods, seasonal sparsity, and new SKU introductions without dividing by zero:
1. **Tier 1 (Seasonal Naive):** Actual sales from $t - 52$ weeks ago (if observed and $> 0$).
2. **Tier 2 (Trailing Moving Average):** Mean actual sales over the trailing 4 weeks ($t-1$ to $t-4$, if $> 0$).
3. **Tier 3 (Category Run-Rate):** Category average sales scaled by SKU price ratio.

In [2]:
metrics_path = settings.artifacts_dir / 'metrics' / 'cv_summary.json'
cv_summary = json.loads(metrics_path.read_text(encoding='utf-8'))

folds_df = pd.DataFrame(cv_summary['folds'])
display(folds_df[['fold', 'train_end_week', 'valid_weeks', 'n_rows', 'wape', 'bias', 'mae', 'rmse']])
print(f"\nOverall Pooled Baseline WAPE: {cv_summary['overall_wape']:.2%}")
print(f"Overall Baseline Forecast Bias: {cv_summary['overall_bias']:+.4f}")

,fold,train_end_week,valid_weeks,n_rows,wape,bias,mae,rmse
0,1,2024-W52,8,40000,0.380051,0.062236,6.245456,11.692543
1,2,2025-W16,8,40000,0.352940,0.043147,6.630525,10.685014
2,3,2025-W40,12,60000,0.390487,-0.025923,8.491412,37.327771



Overall Pooled Baseline WAPE: 37.76%
Overall Baseline Forecast Bias: +0.0146


## 2. Baseline Visual Inspection

We inspect the seasonal-naive baseline prediction against actual historical demand for representative SKUs.

In [3]:
sample_sku = weekly['sku_id'].iloc[0]
sku_data = weekly[weekly['sku_id'] == sample_sku].sort_values('year_week').tail(52)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sku_data['year_week'], y=sku_data['units_sold'], mode='lines+markers', name='Actual Sales'))
fig.update_layout(
    title=f"Actual Demand Trajectory for {sample_sku} (Trailing 52 Weeks)",
    xaxis_title="Year-Week",
    yaxis_title="Units Sold",
    template="plotly_white"
)
fig.show()

## 3. Baseline Gate Target

The baseline establishes the bar that machine learning must beat:
- **Baseline Benchmark WAPE: 37.76%**
- **Fold 1 WAPE: 38.01%**
- **Fold 2 WAPE: 35.29%**
- **Fold 3 WAPE: 39.05%**

Phase 3 ML models must reduce this error across all 3 folds to earn production certification.